# 07 — Advanced RAG Pipeline (80%+ Accuracy Target)

To reach 80% accuracy, we are implementing three major upgrades:
1. **Hybrid Retrieval**: Combining Vector Search (FAISS) with Keyword Search (BM25).
2. **Cross-Encoder Reranking**: Using a powerful reranker to refine retrieval results.
3. **Two-Stage Model**: Loading the PubMedBERT model that was pretrained on artificial data and fine-tuned on expert labels.

## 1. Setup and Imports

In [ ]:
import torch
import numpy as np
import faiss
from datasets import load_from_disk, load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from sklearn.metrics import classification_report, accuracy_score
import tqdm

## 2. Load Documents for Knowledge Base

In [ ]:
# Using the labeled dataset as our knowledge base source
dataset = load_dataset("pubmed_qa", "pqa_labeled")

documents = []
for item in dataset["train"]:
    context = " ".join(item["context"]["contexts"])
    long_answer = item["long_answer"]
    documents.append(context)
    documents.append(long_answer)

documents = list(set(documents)) # Remove duplicates
print(f"Total unique documents in KB: {len(documents)}")

## 3. Initialize Retrieval Models

In [ ]:
print("Initializing Embedder and Reranker...")
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Pre-calculate embeddings for FAISS
print("Encoding documents for Vector Search...")
doc_embeddings = embedder.encode(documents, show_progress_bar=True)
index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(np.array(doc_embeddings))

# Initialize BM25 for Keyword Search
print("Initializing BM25 for Keyword Search...")
tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

## 4. Hybrid Retrieval + Reranking Function

In [ ]:
def advanced_retrieve(query, top_k_initial=20, top_k_final=3):
    # 1. Vector Search (FAISS)
    query_embedding = embedder.encode([query])
    _, faiss_indices = index.search(query_embedding, top_k_initial)
    faiss_results = [documents[i] for i in faiss_indices[0]]
    
    # 2. Keyword Search (BM25)
    tokenized_query = query.lower().split()
    bm25_results = bm25.get_top_n(tokenized_query, documents, n=top_k_initial)
    
    # 3. Combine and Deduplicate
    combined_candidates = list(set(faiss_results + bm25_results))
    
    # 4. Rerank with Cross-Encoder
    pairs = [[query, doc] for doc in combined_candidates]
    scores = reranker.predict(pairs)
    
    # Sort by score and take top_k_final
    reranked_indices = np.argsort(scores)[::-1][:top_k_final]
    final_results = [combined_candidates[i] for i in reranked_indices]
    
    return final_results

## 5. Load the Two-Stage Trained Model

In [ ]:
model_path = "models/pubmedbert_final" # Path to the two-stage model
tokenizer = AutoTokenizer.from_pretrained("microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext")
classifier = AutoModelForSequenceClassification.from_pretrained(model_path).to("mps" if torch.backends.mps.is_available() else "cpu")
classifier.eval()

## 6. Final QA Logic

In [ ]:
def advanced_answer_question(question):
    # Retrieve top 3 documents using hybrid + rerank
    docs = advanced_retrieve(question)
    
    # Augment input
    evidence_text = " [SEP] ".join(docs)
    combined_text = f"Question: {question} [SEP] Evidence: {evidence_text}"
    
    inputs = tokenizer(
        combined_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(classifier.device)
    
    with torch.no_grad():
        outputs = classifier(**inputs)
    
    logits = outputs.logits
    pred = torch.argmax(logits, dim=1).item()
    
    labels = ["yes", "no", "maybe"]
    return {
        "answer": labels[pred],
        "evidence": docs
    }

## 7. Evaluation on Test Set

In [ ]:
test_dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)["test"]

predictions = []
true_labels = []

print("Evaluating Advanced RAG Pipeline...")
for item in tqdm.tqdm(test_dataset):
    result = advanced_answer_question(item["question"])
    predictions.append(result["answer"])
    true_labels.append(item["final_decision"])

print(f"\nFinal Accuracy: {accuracy_score(true_labels, predictions):.4f}")
print("\nClassification Report:")
print(classification_report(true_labels, predictions, target_names=["yes", "no", "maybe"]))